<a href="https://colab.research.google.com/github/MuhammadRhakan/final_project/blob/main/3_Refined_Methods.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import nltk
import kagglehub

import pandas as pd
import numpy as np
import networkx as nx

from kagglehub.datasets import KaggleDatasetAdapter
from sklearn.preprocessing import normalize, StandardScaler, OneHotEncoder, PowerTransformer
from sklearn.metrics import pairwise_distances
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
from nltk import pos_tag, word_tokenize
from nltk.corpus import stopwords, wordnet
from nltk.stem import WordNetLemmatizer
from nltk.stem.porter import *
from scipy.spatial.distance import cdist
from scipy.sparse import csr_matrix


nltk.download('punkt')
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('punkt_tab')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [2]:
course = kagglehub.load_dataset(KaggleDatasetAdapter.PANDAS, "hossaingh/udemy-courses", "Course_info.csv")

<ipython-input-2-193580186>:1: DeprecationWarning: load_dataset is deprecated and will be removed in a future version.
  course = kagglehub.load_dataset(KaggleDatasetAdapter.PANDAS, "hossaingh/udemy-courses", "Course_info.csv")


In [3]:
course = course[course['language'].isin({'English'})]

In [4]:
def missing_values(course):
  null_values = course[(course['num_lectures'] == 0) |
                (course['content_length_min'] == 0) |
                ((course['avg_rating'] == 0) & (course['num_reviews'] > 0))].index
  clean_data = course.drop(index=null_values).dropna()

  return clean_data

In [5]:
course = missing_values(course)

In [6]:
#separate numeric and non-numeric attributes
def attributes(data, shift='avg_rating'):
  categorical = []
  numerical = []

  for i, cat in enumerate(data.select_dtypes(include = ['object', 'bool']).columns.values):
    categorical.append(cat)
  categorical.append(shift)

  for i, num in enumerate(data.select_dtypes(include = 'number').drop(columns='id').columns.values):
    if num != shift:
      numerical.append(num)

  return categorical, numerical

#prepare clean data
def data_cleaning(data, features, par=0.9):
  outliers_indices = set()
  transformer = PowerTransformer(method='box-cox')

  for col in features:
    exclude = data[col].quantile(par)
    outliers = data[data[col] > exclude]
    outliers_indices.update(outliers.index)

  trim = data.drop(index=outliers_indices)
  transformed = trim.copy()

  transformed[['num_subscribers', 'num_reviews', 'num_comments']] = np.log1p(transformed[['num_subscribers', 'num_reviews', 'num_comments']])
  transformed[['price', 'num_lectures', 'content_length_min']] = np.sqrt(transformed[['price', 'num_lectures', 'content_length_min']])

  return trim, transformed

#modularize each feature type
def features_type(data):
  return {
      'semantic': ['title', 'headline'],
      'nominal': ['is_paid', 'category', 'subcategory'],
      'datetime': ['published_time', 'last_update_date'],
      'high_cardinal': 'instructor_name',
      'ordinal': 'avg_rating'}

#feature engineering for high-cardinality categorical data
def calc_smoothed_instructor_rating(data, feature, rating='avg_rating', subscriber='num_subscribers', weight=50):
  data['engagement'] = data[rating] * data[subscriber]

  instructor_stats = data.groupby(feature).agg(
      total_rating=('engagement', 'sum'),
      total_subs=(subscriber, 'sum'))

  instructor_stats['weighted_avg'] = instructor_stats['total_rating'] / instructor_stats['total_subs']
  global_avg = data['engagement'].sum() / data[subscriber].sum()
  instructor_stats['smoothed'] = (
      (instructor_stats['total_subs'] * instructor_stats['weighted_avg'] + weight * global_avg) /
      (instructor_stats['total_subs'] + weight))

  data['instructor_score'] = data[feature].map(instructor_stats['smoothed'])
  data[['avg_rating', 'instructor_score']] = data[['avg_rating', 'instructor_score']].astype('int64')

  return data[['avg_rating', 'instructor_score']]

In [7]:
categorical, numerical = attributes(course)
course_clean, course_clean_scaled = data_cleaning(course, numerical)
types = features_type(course)

In [ ]:
def semantic_preprocessing_batch(batch, features, stop_words, lemmatizer):
    batch = batch[features].copy()
    for col in features:
        batch[col] = batch[col].str.lower()

    for col in features:
        batch[col] = batch[col].apply(lambda text: ' '.join([lemmatizer.lemmatize(word) for word in word_tokenize(text) if word not in stop_words]))

    return batch.apply(lambda row: ' '.join(row), axis=1)


def semantic_preprocessing(data, features, batch_size=5000):
    stop_words = set(stopwords.words('english'))
    lemmatizer = WordNetLemmatizer()
    vectorizer = TfidfVectorizer(max_features=1000, min_df=10, max_df=0.9, ngram_range=(1, 2), dtype=np.float32)

    combined_text = []
    for i in range(0, len(data), batch_size):
      batch = data.iloc[i:i+batch_size]
      processed = semantic_preprocessing_batch(batch, features, stop_words, lemmatizer)
      combined_text.extend(processed)

    tfidf_matrix = vectorizer.fit_transform(combined_text)

    similarity_batches = []
    for i in range(0, tfidf_matrix.shape[0], batch_size):
        batch = tfidf_matrix[i:i+batch_size]
        similarity_batch = cosine_similarity(batch, tfidf_matrix)
        similarity_batches.append(similarity_batch.astype(np.float16))

    return tfidf_matrix, np.vstack(similarity_batches)

In [ ]:
semantic_result, semantic_similarities = semantic_preprocessing(course_clean_scaled, types['semantic'])

In [ ]:
print(f'TF-IDF Result:  {semantic_result.shape[0]} rows x {semantic_result.shape[1]} words')
print(f'Final Matrix:   {semantic_similarities.shape[0]} items x {semantic_similarities.shape[1]} items')

TF-IDF Result:  85480 rows x 1000 words
Final Matrix:   85480 items x 85480 items


In [8]:
def numerical_preprocessing_batch(data, features, batch_size=10000, top_k=100):
  normalized_data = normalize(data[features]).astype(np.float32)
  n_samples = normalized_data.shape[0]

  similarities = np.zeros((n_samples, top_k), dtype=np.float32)
  indices = np.zeros((n_samples, top_k), dtype=int)

  for start in range(0, n_samples, batch_size):
      end = min(start + batch_size, n_samples)
      batch = normalized_data[start:end]

      distances = cdist(batch, normalized_data, metric='euclidean')
      batch_similarities = 1 / (1 + distances)

      top_k_idx = np.argpartition(-batch_similarities, top_k - 1, axis=1)[:, :top_k]

      for i in range(end - start):
          idx = top_k_idx[i]
          sim_vals = batch_similarities[i, idx]
          order = np.argsort(-sim_vals)
          similarities[start + i] = sim_vals[order]
          indices[start + i] = idx[order]

  return similarities, indices

In [9]:
numeric_result, numeric_similarities = numerical_preprocessing_batch(course_clean_scaled, numerical)

In [13]:
numeric_similarities[:5, :5]

array([[    0, 21925, 27156,  1855, 40143],
       [    1,   729, 12158, 16495, 13458],
       [    2, 32546, 17338, 50865, 72854],
       [    3, 57616, 63954, 50831, 16751],
       [    4, 17804,  6880, 28449, 56774]])

In [ ]:
from itertools import combinations

def nominal_preprocessing(data, features, batch_size=5000):
  categorical = data[features].astype(str)

  n_items = len(categorical)
  n_features = len(features)
  similarity_matrix = np.zeros((n_items, n_items))

  for i, j in combinations(range(n_items), 2):
    nij = (categorical.iloc[i] == categorical.iloc[j]).sum()
    ni = n_features
    nj = n_features
    ds = (2 * nij) / (ni + nj)

    similarity_matrix[i, j] = ds
    similarity_matrix[j, i] = ds

  np.fill_diagonal(similarity_matrix, 1.0)

  return pd.DataFrame(similarity_matrix, index=data.index, columns=data.index)

nominal_preprocessing(course_clean, types['nominal'])

In [ ]:
def ordinal_preprocessing(data, n_neighbors=10):
  ordinal_data = data.rank(axis=0, method='average')

  normalized_data = normalize(ordinal_data, norm='l2', axis=1)
  nbrs = NearestNeighbors(n_neighbors=n_neighbors+1, metric='cosine', n_jobs=-1)
  nbrs.fit(normalized_data)

  distances, indices = nbrs.kneighbors(normalized_data)
  cosine_similarities = 1 - distances

  return cosine_similarities[:, 1:], indices[:, 1:]